In [29]:
import pandas as pd
import numpy as np

In [30]:
df = pd.read_csv('/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/silver_standard_dataset.csv')

In [31]:
# Split the 'label' column into 'hierarchical_categories' and 'scenarios' columns
import re

def extract_hierarchical_categories(label):
    match = re.search(r'\((.*?)\)', label)  
    return match.group(1) if match else ''

def extract_scenarios(label):
    match = re.search(r'\)\s*,\s*(.*)', label)
    return match.group(1) if match else ''

# Create the new columns
df['hierarchical_categories'] = df['label'].apply(extract_hierarchical_categories)
df['scenarios'] = df['label'].apply(extract_scenarios)

In [32]:
df.head()

,number,text,label,reasoning,scientific_confidence,max_path_confidence,consistency,hierarchical_categories,scenarios
0,INC0122752,Support - Login/My YouSee/New Customer (first ...,"(1g Self service, 2g Mit YouSee), 3 Login issues","Tag: (1g Self service, 2g Mit YouSee), 3 Login...",0.9790,0.95,1.0,"1g Self service, 2g Mit YouSee",3 Login issues
1,INC0129065,PARENT ID - INC0118835 - Support - YouSee mail...,"(1g Self service, 2g Webmail), 3 technical issues","(1g Self service, 2g Webmail), 3 technical iss...",0.9700,0.92,1.0,"1g Self service, 2g Webmail",3 technical issues
2,INC0144744,Support - Login/My YouSee/Migrated Customer (b...,"(1g Self service, 2g Mit YouSee), 3 Login issues","(1g Self service, 2g Mit YouSee), 3 Login issu...",0.9724,0.92,1.0,"1g Self service, 2g Mit YouSee",3 Login issues
3,INC0122754,Support - Login/My YouSee/Dawn Native Customer...,"(1g Self service, 2g Mit YouSee), 3 Login issues","Tag: (1g Self service, 2g Mit YouSee), 3 Login...",0.9754,0.93,1.0,"1g Self service, 2g Mit YouSee",3 Login issues
4,INC0129810,Support - TV/Video on demand/SmartTV The Custo...,"(1g Self service, 2g YS-play), 3 Performance i...","Tag: (1g Self service, 2g YS-play), 3 Performa...",0.8142,0.90,0.8,"1g Self service, 2g YS-play",3 Performance issues


In [33]:
df['scenarios'].value_counts()

scenarios
3 Quote error                 3481
3 Login issues                3168
3 technical issues            2819
3 Order activation            1805
3 Missing rights to access     749
3 Billing/Invoices             708
3 Termination                  656
3 onsite technician            465
3 Web                          410
3 App                          383
3 Tickets                      348
3 Order confirmation           345
3 Port in                      285
3 CPR issues in Dawn           283
3 Performance issues           282
3 Mix-tv                       275
3 payment method               274
3 Port out                     191
3 product status               179
3 Change ownership             178
3 Barring                      173
3 Third party                  124
3 Shipment                     118
3 relocate                     116
3 CSRD                         107
3 Sikkerhedspakke              103
3 Stock                         92
3 Click & Collect               87
3 Return/r

In [34]:
# =========================
# CONFIG
# =========================
RANDOM_STATE = 42
LEXICON_SIZE = 5000          # retrieval pool size (few-shot / TF-IDF base)
DISTILLATION_CAP = 25000     # set to None if you want "all remaining"
GROUP_COL = "scenarios"      # you already have this column in df
ID_COL = "number"            # your stable ticket id (INC....)

# =========================
# SAFETY CHECKS
# =========================
required_cols = {ID_COL, "text", GROUP_COL}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"df is missing required columns: {missing}")

df = df.copy()
df = df.dropna(subset=[ID_COL, "text", GROUP_COL]).reset_index(drop=True)

In [35]:
# =========================
# STRATIFIED SAMPLING HELPERS
# =========================
def stratified_sample(
    data: pd.DataFrame,
    group_col: str,
    n_total: int,
    random_state: int = 42,
    min_per_group: int = 5,
    max_per_group: int = 200,
    id_col: str = None,
) -> pd.DataFrame:
    """
    Stratified sample with:
      - proportional allocation per group
      - min/max per group
      - capped by group availability
      - exact total size adjustment
    """
    if n_total > len(data):
        raise ValueError(f"n_total={n_total} > len(data)={len(data)}")

    counts = data[group_col].value_counts(dropna=False)
    props = counts / counts.sum()

    # initial proportional allocation
    alloc = (props * n_total).round().astype(int)

    # apply min/max constraints (only for groups that exist)
    alloc = alloc.clip(lower=min_per_group, upper=max_per_group)

    # cap by availability
    alloc = alloc.combine(counts, func=min)

    # adjust to exact n_total
    total = int(alloc.sum())
    rng = np.random.RandomState(random_state)

    if total < n_total:
        candidates = alloc[(alloc < counts) & (alloc < max_per_group)].index.tolist()
        i = 0
        while total < n_total and candidates:
            g = candidates[i % len(candidates)]
            if alloc[g] < counts[g] and alloc[g] < max_per_group:
                alloc[g] += 1
                total += 1
            i += 1

    elif total > n_total:
        candidates = alloc[alloc > min_per_group].index.tolist()
        i = 0
        while total > n_total and candidates:
            g = candidates[i % len(candidates)]
            if alloc[g] > min_per_group:
                alloc[g] -= 1
                total -= 1
            i += 1

    # sample within each group
    parts = []
    for g, k in alloc.items():
        k = int(k)
        if k <= 0:
            continue
        grp = data[data[group_col] == g]
        parts.append(grp.sample(n=k, random_state=rng.randint(0, 1_000_000)))

    out = pd.concat(parts, ignore_index=True)

    # final correction to exact size (rare edge cases)
    if len(out) > n_total:
        out = out.sample(n=n_total, random_state=random_state).reset_index(drop=True)
    elif len(out) < n_total:
        need = n_total - len(out)
        if id_col is not None:
            remaining = data[~data[id_col].isin(out[id_col])].copy()
        else:
            remaining = data.drop(out.index, errors="ignore").copy()
        if need > 0:
            out = pd.concat(
                [out, remaining.sample(n=min(need, len(remaining)), random_state=random_state)],
                ignore_index=True
            )
        out = out.reset_index(drop=True)

    return out

In [36]:
# =========================
# 1) BUILD LEXICON (RETRIEVAL POOL)
# =========================
lexicon_df = stratified_sample(
    df,
    group_col=GROUP_COL,
    n_total=LEXICON_SIZE,
    random_state=RANDOM_STATE,
    min_per_group=5,     # ensures tail scenarios show up
    max_per_group=200,   # prevents big scenarios from dominating
    id_col=ID_COL
)

# =========================
# 2) BUILD DISTILLATION SET (NO LEAKAGE)
# =========================
lexicon_ids = set(lexicon_df[ID_COL])
distill_df = df[~df[ID_COL].isin(lexicon_ids)].copy()

# Optional: cap distillation to exactly 25k (still stratified)
if DISTILLATION_CAP is not None and len(distill_df) > DISTILLATION_CAP:
    distill_df = stratified_sample(
        distill_df,
        group_col=GROUP_COL,
        n_total=DISTILLATION_CAP,
        random_state=RANDOM_STATE,
        min_per_group=0,       # don't force tail here
        max_per_group=10_000,  # effectively no cap
        id_col=ID_COL
    )

In [37]:
# =========================
# 3) SANITY CHECKS
# =========================
overlap = set(lexicon_df[ID_COL]).intersection(set(distill_df[ID_COL]))
assert len(overlap) == 0, f"Leakage detected! Overlap size: {len(overlap)}"

print("✅ Lexicon size:", len(lexicon_df))
print("✅ Distillation size:", len(distill_df))

print("\nLexicon scenario distribution (top 15):")
print(lexicon_df[GROUP_COL].value_counts().head(15))

print("\nDistillation scenario distribution (top 15):")
print(distill_df[GROUP_COL].value_counts().head(15))

# =========================
# 4) REMOVE SPLIT COLUMNS BEFORE SAVE
# =========================
for col in ["scenarios", "hierarchical_categories"]:
    if col in lexicon_df.columns:
        lexicon_df = lexicon_df.drop(columns=[col])
    if col in distill_df.columns:
        distill_df = distill_df.drop(columns=[col])

✅ Lexicon size: 5000
✅ Distillation size: 13461

Lexicon scenario distribution (top 15):
scenarios
3 Quote error                 200
3 Login issues                200
3 technical issues            200
3 Order activation            200
3 Missing rights to access    200
3 Billing/Invoices            200
3 Termination                 200
3 onsite technician           200
3 Web                         200
3 App                         200
3 Tickets                     200
3 Order confirmation          200
3 Port in                     191
3 CPR issues in Dawn          191
3 Performance issues          190
Name: count, dtype: int64

Distillation scenario distribution (top 15):
scenarios
3 Quote error                 3281
3 Login issues                2968
3 technical issues            2619
3 Order activation            1605
3 Missing rights to access     549
3 Billing/Invoices             508
3 Termination                  456
3 onsite technician            265
3 Web                        

In [40]:
# =========================
# 6) CROSS-CHECK: VERIFY SELECTED EXAMPLES ARE NOT IN LEXICON
# =========================

# Read CSV with examples that should NOT be in the lexicon
exclusion_csv_path = "/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/human_review_random_50.csv"
exclusion_df = pd.read_csv(exclusion_csv_path)

# Extract ticket IDs (number column) from the exclusion CSV
# Use the same ID_COL as defined above ("number")
if ID_COL in exclusion_df.columns:
    exclusion_ids = set(exclusion_df[ID_COL])
else:
    # If different column name, use first column as fallback
    exclusion_ids = set(exclusion_df.iloc[:, 0])

# Check for any overlap between exclusion list and lexicon
overlap_with_lexicon = exclusion_ids.intersection(set(lexicon_df[ID_COL]))

if len(overlap_with_lexicon) == 0:
    print("✅ SUCCESS: None of the exclusion examples are in the lexicon")
    print(f"   - Checked {len(exclusion_ids)} exclusion examples")
    print(f"   - Lexicon size: {len(lexicon_df)}")
else:
    # 1. DELETE: Filter the lexicon to keep only rows NOT in the exclusion set
    # The ~ symbol is the "NOT" operator in pandas
    lexicon_df = lexicon_df[~lexicon_df[ID_COL].isin(overlap_with_lexicon)]
    
    print(f"✅ Cleaned Lexicon size: {len(lexicon_df)}")


✅ Cleaned Lexicon size: 4982


In [41]:
# =========================
# 5) SAVE (OPTIONAL)
# =========================
lexicon_df.to_csv("/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/lexicon_scenarios_5000.csv", index=False)
#distill_df.to_csv("/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/distill_scenarios_25000.csv", index=False)